# Feeling Engine · Interactive Demo

**What you'll do in this notebook:** take a mechanism-arc that's already been mined from a YouTube video and run the Feeling Engine pipeline on it — *without* running TRIBE inference yourself.

Four arcs ship with the repo and are preloaded here:

| Arc | Channel | Duration | Content type |
|---|---|---|---|
| Steve Jobs — Stanford Death Pivot | Stanford | 60s | Speech excerpt |
| Evolution of Homes | @PastMorph | 4m45 | Visual / music |
| Your Life as Every Bloods Rank | @Iceberger99 | 7m29 | POV narrative |
| The 90 Day Trap | @the_corporateplaybook | 9m24 | Adversarial explainer |

Each bundle includes: TRIBE-predicted brain activations (per timestep, 7 HCP categories), a mechanism-form arc, sequence matches, and a transcript.

**What this notebook does NOT do:** run TRIBE inference. That requires GPU + a gated HuggingFace model. If you want to analyze your own content end-to-end, see the `examples/analyze_speech.py` CLI in the repo.

> This notebook produces output derived from TRIBE predictions, which are CC BY-NC 4.0 (Meta FAIR). Use for research / non-commercial purposes only. The mechanism-form arcs and Feeling Engine code are MIT.


## 1. Setup

Clone the repo and install in editable mode. No API keys or tokens needed for this notebook.

In [ ]:
!git clone --depth 1 https://github.com/humansourcecode/feelingengine.git
%cd feelingengine
!pip install -q -e .


## 2. Load a bundled arc

We'll start with the **Steve Jobs — Stanford Death Pivot** because it's short (60s) and has pre-rendered brain hemispheres. Swap the filename to explore other arcs.

In [ ]:
import json

ARC_FILES = {
    'jobs':     'examples/arcs/steve_jobs_death_pivot.json',
    'pastmorph':'examples/arcs/pastmorph_evolution_of_homes.json',
    'iceberger':'examples/arcs/iceberger_bloods_rank.json',
    'corporate':'examples/arcs/corporate_playbook_90_day_trap.json',
}

with open(ARC_FILES['jobs']) as f:
    bundle = json.load(f)

src = bundle['source']
print(f"Title:     {src['title']}")
print(f"Channel:   {src.get('channel_handle') or src.get('channel')}")
print(f"Duration:  {src['duration_sec']}s")
print(f"YouTube:   {src['url']}")
print()
print(f"Timesteps:     {len(bundle['tribe_profiles'])}")
print(f"Pre-computed absolute-mode mechanisms:  {bundle['counts']['n_labels_absolute']}")
print(f"Pre-computed σ-mode mechanisms:         {bundle['counts']['n_labels_sigma']}")


## 3. What's in a TRIBE profile

Each timestep is one second. Each `categories` dict is the 7-region HCP aggregation of TRIBE's per-vertex predictions. Values are signed — positive means above the predicted brain baseline, negative means below.

In [ ]:
profile_at_peak = max(
    bundle['tribe_profiles'],
    key=lambda p: sum(abs(v) for v in p['categories'].values())
)
print(f"Peak timestep: t={profile_at_peak['timestep']}s")
for axis, value in sorted(profile_at_peak['categories'].items(),
                           key=lambda kv: -abs(kv[1])):
    bar = '█' * int(abs(value) * 40)
    print(f"  {axis:<15} {value:+.3f}  {bar}")


## 4. Run mechanism detection — absolute mode vs σ mode

**Absolute mode** uses fixed thresholds empirically tuned against a speech-heavy baseline (the Steve Jobs clip, in fact).

**σ mode** normalizes thresholds against *this video's* axis distribution — it asks "how notable is this moment relative to the rest of this piece?" Much better for cross-content-type analysis.

In [ ]:
from feeling_engine.mechanisms.api import detect_mechanisms, detect_sequences
from feeling_engine.mechanisms.tier1_detectors import compute_axis_stats

profiles = bundle['tribe_profiles']
transcript = bundle['transcript']

# Absolute mode
arc_abs = detect_mechanisms(tribe_categories=profiles, transcript=transcript)

# σ mode (per-video normalized)
axis_stats = compute_axis_stats(profiles)
arc_sig = detect_mechanisms(
    tribe_categories=profiles, transcript=transcript, axis_stats=axis_stats,
)

print(f"Absolute mode: {len(arc_abs)} mechanism applications")
print(f"σ mode:        {len(arc_sig)} mechanism applications")
print(f"Difference:    {len(arc_sig) - len(arc_abs):+d}")


### Label frequency comparison

Which mechanisms fire most under each mode? On quieter content (visual/music), σ mode reveals mechanisms that absolute mode silently misses.

In [ ]:
from collections import Counter

def label_counts(arc):
    return Counter(a.label for a in arc)

abs_c = label_counts(arc_abs)
sig_c = label_counts(arc_sig)
all_labels = sorted(set(abs_c) | set(sig_c))

print(f"{'label':<25} {'abs':>5} {'σ':>5} {'Δ':>5}")
print(f"{'-'*25} {'-'*5} {'-'*5} {'-'*5}")
for lbl in all_labels:
    a, s = abs_c.get(lbl, 0), sig_c.get(lbl, 0)
    print(f"{lbl:<25} {a:>5} {s:>5} {s-a:>+5}")


## 5. Named sequence detection

Individual mechanism firings compose into named narrative sequences (joke-structure, intimacy-deepening, contemplation-spiral, …). 10 sequences ship in v1.

In [ ]:
sequences = detect_sequences(arc_sig)

if sequences:
    for seq in sequences:
        kind = '(partial match)' if seq.partial else '(full match)'
        chain = ' → '.join(seq.matched_labels)
        print(f"  • {seq.name}  t={seq.start_sec:.0f}s → {seq.end_sec:.0f}s  {kind}")
        print(f"     chain: {chain}")
else:
    print("No named sequences matched this clip.")


## 6. Visualize the brain activity timeline

Each of the 7 HCP categories plotted over the content's duration. This is the same data the Inspector renders as a live chart; here we just plot it with matplotlib.

In [ ]:
import matplotlib.pyplot as plt

AXES = ['interoception','core_affect','regulation','reward','memory','social','language']
colors = {'interoception':'#f97316','core_affect':'#eab308','regulation':'#22d3ee',
          'reward':'#a78bfa','memory':'#6ee7b7','social':'#fb7185','language':'#60a5fa'}

fig, ax = plt.subplots(figsize=(10, 4))
t = [p['timestep'] for p in profiles]
for axis in AXES:
    ax.plot(t, [p['categories'][axis] for p in profiles],
            color=colors[axis], label=axis, linewidth=1)
ax.axhline(0, color='#666', linewidth=0.5, linestyle='--')
ax.set_xlabel('timestep (s)')
ax.set_ylabel('activation')
ax.set_title(src['title'])
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 7. Compare arcs across the full library

Mechanisms vary dramatically by content type. σ mode's key property: firing-rate *per second* stays in a comparable range across all four content types, while absolute mode's rate varies 10×.

In [ ]:
print(f"{'arc':<35} {'dur':>6} {'abs/s':>7} {'σ/s':>7}")
print(f"{'-'*35} {'-'*6} {'-'*7} {'-'*7}")
for key, path in ARC_FILES.items():
    with open(path) as f:
        b = json.load(f)
    dur = b['source']['duration_sec']
    ar = b['counts']['n_labels_absolute'] / dur
    sr = b['counts']['n_labels_sigma']    / dur
    name = b['source']['title'][:35]
    print(f"{name:<35} {int(dur):>6} {ar:>7.2f} {sr:>7.2f}")


## 8. Next steps

- **Run the visual inspector locally** — the same data rendered as an interactive timeline + brain hemispheres + scrubber. From the repo root: `cd examples && python3 -m http.server 8080`, then open `http://localhost:8080/inspector/`.
- **Analyze your own content** — see `examples/analyze_speech.py` for the full CLI (requires Modal + HuggingFace access for TRIBE inference).
- **Mine more outlier arcs** — `feeling_engine/mining/arc_miner.py` takes any YouTube URL → arc-library row. Grow the library.
- **Read the methodology** — `docs/methodology.md` · `docs/mechanism_labels.md` · `docs/detector_validation.md`.

Questions / issues: [github.com/humansourcecode/feelingengine/issues](https://github.com/humansourcecode/feelingengine/issues)
